# Uni-MuMER - Kaggle 2xT4 + DagsHub

Train QLoRA, theo dõi metric trực tiếp bằng MLflow và lưu toàn bộ artifact lên DagsHub.

In [ ]:
# 1. Cấu hình
import json
import os
import uuid

from kaggle_secrets import UserSecretsClient

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"
PIP = f"{ENV_DIR}/bin/pip"
LLAMAFACTORY = f"{ENV_DIR}/bin/llamafactory-cli"
TRAIN_CONFIG = "train/Uni-MuMER-train.yaml"
OUTPUT_DIR = "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora"

DAGSHUB_USERNAME = "NhatPot"
DAGSHUB_REPO = "test-unimer"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
RUN_UUID = uuid.uuid4().hex
DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
if not DAGSHUB_TOKEN:
    raise RuntimeError("Kaggle Secret DAGSHUB_TOKEN is missing")

os.environ.pop("PYTHONPATH", None)
os.environ.update({
    "PROJECT_DIR": PROJECT_DIR,
    "CONDA_DIR": CONDA_DIR,
    "ENV_DIR": ENV_DIR,
    "PYTHON": PYTHON,
    "PIP": PIP,
    "LLAMAFACTORY": LLAMAFACTORY,
    "TRAIN_CONFIG": TRAIN_CONFIG,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_UUID": RUN_UUID,
    "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow",
    "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
    "MLFLOW_FLATTEN_PARAMS": "TRUE",
    "MLFLOW_TAGS": json.dumps({
        "run_uuid": RUN_UUID,
        "source": "kaggle",
        "task": "sft",
        "dataset": "parquet_crohme_train",
    }),
})

print(f"Run UUID: {RUN_UUID}")
print(f"MLflow: {os.environ['MLFLOW_TRACKING_URI']}")

In [ ]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$PYTHON" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -f -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version

In [ ]:
%%bash
# 3. Lấy source code
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi

git -C "$PROJECT_DIR" rev-parse --short HEAD

In [ ]:
%%bash
# 4. Cài dependency
set -euo pipefail
cd "$PROJECT_DIR"

"$PIP" install -q -r requirements.txt
"$PIP" install -q -e train/LLaMA-Factory
"$PYTHON" -c "import torch, mlflow; print('GPU:', torch.cuda.get_device_name(0)); print('MLflow:', mlflow.__version__)"

In [ ]:
%%bash
# 5. Kiểm tra DagsHub trước khi train
set -euo pipefail
cd "$PROJECT_DIR"

"$PYTHON" scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"

In [ ]:
%%bash
# 6. Training
set -euo pipefail
cd "$PROJECT_DIR"

MPLBACKEND=Agg "$LLAMAFACTORY" train "$TRAIN_CONFIG" "run_name=uni-mumer-${RUN_UUID:0:8}"

In [ ]:
%%bash
# 7. Upload và xác minh artifact
set -euo pipefail
cd "$PROJECT_DIR"

"$PYTHON" scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$TRAIN_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR"